# RNN Model Multivariate

In this section we implement multivariate forecasting using the RNN Model with the **TimeSeriesDatasetVectorizedExog** approach.

The RNN (Recurrent Neural Network) Forecaster is the same univariate model used in the univariate approach, but extended to multivariate forecasting through efficient batching. Instead of processing series individually, **TimeSeriesDatasetVectorizedExog** batches all 1502 series together, allowing the univariate model to train on multiple series simultaneously with exogenous features (GDP, CPI, Interest Rate).

With this approach the model architecture remains unchanged - we simply reshape the data to process all series in parallel, achieving faster training while incorporating exogenous variables.

**Layer Breakdown:**

- **RNN Layers:** 2 stacked RNN layers with Tanh activation
- **Hidden Size:** 64 units per layer
- **Dropout:** Applied between RNN layers (if >1 layer) and before final output
- **Output Layer:** Single fully connected layer producing 1-step forecast

In [1]:
import torch
import torch.nn as nn

## Model

In [2]:
class RNNForecaster(nn.Module):
    """
    Vanilla RNN model for MULTIVARIATE time series forecasting.
    Architecture: RNN -> Dropout -> RNN -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    
    Simpler than LSTM - no cell state, only hidden state.
    Faster training but may struggle with long-term dependencies.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: RNN hidden dimension
            num_layers: Number of RNN layers
            dropout: Dropout rate
        """
        super(RNNForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # RNN layers (using Tanh activation by default)
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'  # Can also use 'relu'
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # RNN forward pass
        # rnn_out: (batch_size, seq_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size)
        rnn_out, h_n = self.rnn(x)
        
        # Take the output from the last time step
        last_output = rnn_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out

## Model Results without Exogenous Features

#### Optuna Hyperparameter Search Results 

| Number | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (s) |
|--------|-----------------|------------|---------|-------------|---------------|------------|--------------|
| 0 | 0.1234 | 16 | 0.202 | 256 | 0.000386 | 2 | 7.647 |
| 1 | 0.1928 | 16 | 0.147 | 32 | 0.000189 | 2 | 0.916 |
| 2 | 0.1557 | 4 | 0.422 | 32 | 0.000199 | 2 | 2.394 |
| 3 | 0.1882 | 8 | 0.147 | 32 | 0.000107 | 2 | 1.240 |
| 4 | 0.1291 | 4 | 0.227 | 32 | 0.002757 | 3 | 3.114 |




### Best Hyperparameters

Best Hyperparameters:
- learning_rate: 0.00039
- batch_size: 16
- num_layers: 2
- hidden_size: 256
- dropout: 0.20153

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/rnn/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/rnn/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

    
![Fold 3 Results](./img/multivariate/rnn/fold3/fold_results.png)


### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|---------|---------|
| Fold 1 | 65033.20 | 255.02 | 105.17 | 0.8711 | 83.86% |
| Fold 2 | 59006.86 | 242.91 | 108.03 | 0.8697 | 90.52% |
| Fold 3 | 53933.09 | 232.23 | 93.24 | 0.8891 | 55.90% |
| **Average** | **59324.38 ± 5556.86** | **243.39 ± 11.40** | **102.15 ± 7.84** | **0.8766 ± 0.0108** | **76.76% ± 18.37%** |


#### SMAPE Distribution Accross Folds

| SMAPE Range | Average % of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 7.1% ± 7.1% | 106 |
| 10-20% | 11.0% ± 1.4% | 166 |
| 20-30% | 13.2% ± 1.3% | 198 |
| 30-40% | 10.2% ± 0.6% | 153 |
| >40% | 58.5% ± 10.4% | 879 |

**Comparison with Baseline:**

The RNN multivariate model achieves an average SMAPE of 76.76% ± 18.37%, which is **4.50 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). While the baseline performs slightly better on average, the RNN model shows significantly more variation across folds (standard deviation: 18.37% vs 7.06%), indicating less consistent performance. However, the RNN demonstrates strong performance in Fold 3 (55.90% SMAPE), suggesting it can capture certain temporal patterns effectively under specific conditions.

## Model Results with Exogenous Features

#### Optuna Hyperparameter Search Results

| Number | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration (s) |
|--------|-----------------|------------|---------|-------------|---------------|------------|--------------|
| 0 | 0.4507 | 16 | 0.467 | 32 | 0.001072 | 2 | 0.451 |
| 1 | 0.3140 | 8 | 0.120 | 256 | 0.000924 | 1 | 1.336 |
| 2 | 0.3488 | 8 | 0.481 | 128 | 0.004272 | 1 | 0.928 |
| 3 | 0.3982 | 4 | 0.277 | 256 | 0.000136 | 2 | 2.044 |
| 4 | 0.3979 | 4 | 0.374 | 32 | 0.008280 | 3 | 1.086 |

### Best Hyperparameters

Parameters:
  - learning_rate: 0.0009
  - batch_size: 8
  - num_layers: 1
  - hidden_size: 256
  - dropout: 0.11960


#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/multivariate/rnn_exog/fold1/fold_results.png)


#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/multivariate/rnn_exog/fold2/fold_results.png)


#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30
    
![Fold 3 Results](./img/multivariate/rnn_exog/fold3/fold_results.png)

### Fold Results 

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|---------|---------|
| Fold 1 | 268096.69 | 517.78 | 469.42 | 0.4687 | 137.12% |
| Fold 2 | 96562.47 | 310.75 | 234.98 | 0.7868 | 123.31% |
| Fold 3 | 63557.92 | 252.11 | 97.96 | 0.8693 | 80.31% |
| **Average** | **142739.03 ± 109809.98** | **360.21 ± 139.57** | **267.45 ± 187.85** | **0.7083 ± 0.2115** | **113.58% ± 29.62%** |

### Average SMAPE Distribution Accross Folds 

| SMAPE Range | Average % of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 2.2% ± 0.9% | 32 |
| 10-20% | 7.3% ± 3.7% | 110 |
| 20-30% | 8.4% ± 4.3% | 126 |
| 30-40% | 6.4% ± 3.0% | 96 |
| >40% | 75.8% ± 11.8% | 1138 |

**Comparison with Baseline:**

The RNN multivariate model with exogenous features achieves an average SMAPE of 113.58% ± 29.62%, which is **41.32 percentage points higher** than the baseline 3-month rolling average (72.26% ± 7.06%). The baseline significantly outperforms this model, and the RNN with exogenous features shows considerably more variation across folds (standard deviation: 29.62% vs 7.06%), indicating highly inconsistent performance. While the model achieves reasonable performance in Fold 3 (80.31% SMAPE), its performance in Folds 1 and 2 (137.12% and 123.31% respectively) is substantially worse than the baseline. This suggests that the addition of exogenous features actually degraded model performance, possibly due to overfitting or difficulty in learning the complex relationships between exogenous variables and the target series.